In [23]:
import pandas as pd
PATH = "../data/raw/senior_living_emails.csv"
df = pd.read_csv(PATH, keep_default_na=False, na_values=[""])
df.shape

(825, 6)

(825, 6)

In [24]:
df.head()

,email_id,date,sender,subject,body,community_type
0,544,2026-04-11,marcus@communities.net,Memory care you can trust in Austin,Join our free caregiver support seminar on dem...,Memory Care
1,627,2026-04-05,david@seniorcare.com,"A safe, secure neighbourhood for memory loss",Join our free caregiver support seminar on dem...,Memory Care
2,176,2026-08-26,david@livingwell.org,Care plans tailored to your loved one,Choosing care for a parent is hard. At Sunrise...,Assisted Living
3,372,2026-04-01,elena@communities.net,Assisted living in Atlanta: help when you need it,Choosing care for a parent is hard. At Riverbe...,Assisted Living
4,145,2026-09-05,tom@eldercircle.com,Personalized care and support at Brookfield Court,Choosing care for a parent is hard. At Brookfi...,Assisted Living


In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 825 entries, 0 to 824
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   email_id        825 non-null    int64
 1   date            825 non-null    str  
 2   sender          825 non-null    str  
 3   subject         782 non-null    str  
 4   body            814 non-null    str  
 5   community_type  794 non-null    str  
dtypes: int64(1), str(5)
memory usage: 38.8 KB


In [26]:
df.isna().sum()

email_id           0
date               0
sender             0
subject           43
body              11
community_type    31
dtype: int64

In [27]:
df["community_type"].value_counts(dropna=False).head(25)

community_type
Independent Living     189
Assisted Living        154
Other/Mixed             95
Memory Care             63
IL                      34
NaN                     31
independent_living      28
Independent Living      27
assisted living         26
independent living      25
Assisted Living         23
other/mixed             18
Assisted-Living         16
N/A                     16
AL                      14
memory_care             14
other                   12
memory care             11
Memory Care             10
Mixed                   10
MC                       9
Name: count, dtype: int64

In [28]:
na_rows = df[df["community_type"] == "N/A"]
print(len(na_rows))
for _, r in na_rows.head(5).iterrows():
    print(r["subject"], "|", r["body"][:200] if isinstance(r["body"], str) else r["body"])
    print("---")

16
Independent, assisted and memory care in one campus | Hello from Evergreen Gardens in Columbus, OH. Our campus offers independent living, assisted living and memory care so residents can age in place. Contact Linda for pricing, starting at .
---
Full continuum of care at Northgate Landing | Northgate Landing newsletter: congratulations to our staff of the year, holiday schedule, volunteer opportunities and upcoming resident events. Questions? Email Jennifer.
---
Full continuum of care at Golden Oaks | Golden Oaks newsletter: congratulations to our staff of the year, holiday schedule, volunteer opportunities and upcoming resident events. Questions? Email Tom.
---
Independent, assisted and memory care in one campus | Hi, I'm Priya, a placement advisor. We help families compare senior living options across Tampa, FL and nearby areas, including 55+ communities, care homes and respite stays. No fee to families.
---
Silver Birch newsletter - November updates | Hi, I'm Tom, a placement adv

In [29]:
na_rows = df[df["community_type"] == "N/A"]
body = na_rows["body"].fillna("").str.lower()

print("Mention independent living:", body.str.contains("independent living").sum())
print("Mention assisted living:   ", body.str.contains("assisted living").sum())
print("Mention memory care:       ", body.str.contains("memory care").sum())
print("Total N/A rows:", len(na_rows))

Mention independent living: 1
Mention assisted living:    1
Mention memory care:        1
Total N/A rows: 16


In [30]:
clean = (
    df["community_type"]
    .str.strip()                          # remove invisible spaces
    .str.lower()                          # unify case
    .str.replace(r"[_\-]", " ", regex=True)   # independent_living / Assisted-Living -> spaces
)
clean.value_counts(dropna=False)

community_type
independent living    269
assisted living       219
other/mixed           113
memory care            98
il                     34
NaN                    31
n/a                    16
al                     14
other                  12
mixed                  10
mc                      9
Name: count, dtype: int64

In [31]:
mapping = {
    "independent living": "Independent Living",
    "il": "Independent Living",
    "assisted living": "Assisted Living",
    "al": "Assisted Living",
    "memory care": "Memory Care",
    "mc": "Memory Care",
    "other/mixed": "Other/Mixed",
    "other": "Other/Mixed",
    "mixed": "Other/Mixed",
    "n/a": "Other/Mixed",
}

df["label"] = clean.map(mapping)

# Anything not covered by the mapping shows up here:
unmapped = df.loc[df["label"].isna() & df["community_type"].notna(), "community_type"]
unmapped.value_counts()

Series([], Name: count, dtype: int64)

In [32]:
df["label"].value_counts(dropna=False)

label
Independent Living    303
Assisted Living       233
Other/Mixed           151
Memory Care           107
NaN                    31
Name: count, dtype: int64

In [33]:
print("Unlabeled:", df["label"].isna().sum())

labeled = df.dropna(subset=["label"]).copy()
labeled.shape

Unlabeled: 31


(794, 7)

In [34]:
dup_cols = ["date", "sender", "subject", "body"]   # not email_id: copies have new ids
print("Duplicates:", labeled.duplicated(subset=dup_cols).sum())

labeled = labeled.drop_duplicates(subset=dup_cols, keep="first").reset_index(drop=True)
labeled.shape

Duplicates: 24


(770, 7)

In [35]:
counts = labeled["label"].value_counts()
print(counts)
print("Majority-class baseline accuracy:", round(counts.max() / counts.sum(), 3))

label
Independent Living    293
Assisted Living       223
Other/Mixed           151
Memory Care           103
Name: count, dtype: int64
Majority-class baseline accuracy: 0.381


In [36]:
labeled["date"] = pd.to_datetime(labeled["date"])
labeled["month"] = labeled["date"].dt.month

body = labeled["body"].fillna("")
labeled["body_length"] = body.str.len()
labeled["mentions_price"] = body.str.contains(r"\$|USD|CAD", regex=True)

labeled[["body_length", "mentions_price", "month"]].describe(include="all")

,body_length,mentions_price,month
count,770.000000,770,770.000000
unique,NaN,2,NaN
top,NaN,True,NaN
freq,NaN,614,NaN
mean,221.351948,NaN,5.607792
std,42.410797,NaN,2.778225
min,0.000000,NaN,1.000000
25%,201.000000,NaN,3.000000
50%,223.000000,NaN,6.000000
75%,253.750000,NaN,8.000000


In [37]:
labeled.groupby("label")[[
    "body_length",
    "mentions_price",
    "month"
]].mean().round(2)

,body_length,mentions_price,month
label,,,
Assisted Living,227.09,0.93,5.50
Independent Living,233.82,0.92,5.67
Memory Care,225.45,0.91,5.45
Other/Mixed,185.90,0.29,5.76


In [38]:
labeled.to_csv("../data/processed/emails_clean.csv", index=False)

In [39]:
print((labeled["body_length"] == 0).sum())
labeled.groupby("label")["mentions_price"].mean().round(2)

11


label
Assisted Living       0.93
Independent Living    0.92
Memory Care           0.91
Other/Mixed           0.29
Name: mentions_price, dtype: float64

In [40]:
both_empty = (labeled["body"].isna() | (labeled["body"] == "")) & (labeled["subject"].isna() | (labeled["subject"] == ""))
print(both_empty.sum())

0
